In [1]:
print("You can do it!")

You can do it!


In [2]:
import duckdb
import pandas as pd

#Load Fashionable sales report and add low_memory flag to be able to read chucks of different data types in the same column.
df = pd.read_csv("/Users/TRTim/PyCharmMiscProject/FashionableSalesReport.csv", low_memory=False)

# print(df.head(50))
df.info(verbose=True)
#Column cleanup
#New naming convention with underscores and lowercase words. Prevents query binding stage error too.
df.columns = df.columns.str.strip()
df.columns = df.columns.str.replace("-", "_")
df.columns = df.columns.str.replace(" ", "_")
df.columns = df.columns.str.lower()

# def data_cleanup(df):
#
#     df.columns = (
#         df.columns
#         .str.strip()
#         .str.replace("-", "_")
#         .str.replace(" ", "_")
#         .str.lower()
#     )
#     return df
#
# df = data_cleanup(df)
# print(df.head())

#Row cleanup
#Incosistent casting inside the rows
df['date'] = pd.to_datetime(df['date'], format='%m-%d-%y')
df['status'] = df['status'].str.strip().str.lower()
df['category'] = df['category'].str.strip().str.lower().str.replace(" ", "_").str.rstrip(".")
df['ship_city']= df['ship_city'].str.strip().str.lower().str.replace(" ", "_").str.rstrip(".")
df['ship_state']= df['ship_state'].str.strip().str.lower().str.replace(" ", "_").str.rstrip(".")

#Connect to database, duckDB
# ducky = duckdb.connect()

#Register the dataframe as a table to bypass incompatible function argument error
# ducky.register("sales_report", df)

#Simple query to make sure clean data is being read
displayQuery = duckdb.sql("SELECT * FROM df LIMIT 6").df()
print(displayQuery)

<class 'pandas.DataFrame'>
RangeIndex: 128975 entries, 0 to 128974
Data columns (total 24 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   index               128975 non-null  int64  
 1   Order ID            128975 non-null  str    
 2   Date                128975 non-null  str    
 3   Status              128975 non-null  str    
 4   Fulfilment          128975 non-null  str    
 5   Sales Channel       128975 non-null  str    
 6   ship-service-level  128975 non-null  str    
 7   Style               128975 non-null  str    
 8   SKU                 128975 non-null  str    
 9   Category            128975 non-null  str    
 10  Size                128975 non-null  str    
 11  ASIN                128975 non-null  str    
 12  Courier Status      122103 non-null  str    
 13  Qty                 128975 non-null  int64  
 14  currency            121180 non-null  str    
 15  Amount              121180 non-null  float64


In [6]:
#Schema of the dataset
# 24 columns with column types
# 128975 rows
# ~3095400 data entries
schema = duckdb.sql("DESCRIBE df").df()
print(schema)

           column_name column_type null   key default extra
0                index      BIGINT  YES  None    None  None
1             order_id     VARCHAR  YES  None    None  None
2                 date   TIMESTAMP  YES  None    None  None
3               status     VARCHAR  YES  None    None  None
4           fulfilment     VARCHAR  YES  None    None  None
5        sales_channel     VARCHAR  YES  None    None  None
6   ship_service_level     VARCHAR  YES  None    None  None
7                style     VARCHAR  YES  None    None  None
8                  sku     VARCHAR  YES  None    None  None
9             category     VARCHAR  YES  None    None  None
10                size     VARCHAR  YES  None    None  None
11                asin     VARCHAR  YES  None    None  None
12      courier_status     VARCHAR  YES  None    None  None
13                 qty      BIGINT  YES  None    None  None
14            currency     VARCHAR  YES  None    None  None
15              amount      DOUBLE  YES 

In [8]:
#Preliminary data assessment:

#Orders taken between March 31 2022 to June 29 2022 (inclusive/ all days inbetween accounted for)
date = duckdb.sql("SELECT DISTINCT date FROM df ORDER BY date ASC").df()
print(date)

#Distribution of shipments based on status. Most common: Shipped. Least common: Shipped-Damaged.
displayStatusCount = duckdb.sql("SELECT status, COUNT(*) AS count_status FROM df GROUP BY status ORDER BY COUNT(*) DESC").df()
print(displayStatusCount)

#Distribution of sizes. Most popular: M and L. Least popular: free and 4XL.
displaySizeCount = duckdb.sql("SELECT size, COUNT(*) AS count_size FROM df GROUP BY size ORDER BY COUNT(*) DESC").df()
print(displaySizeCount)

#Distribution of types of clothing. Most popular: Sets. Least popular: Dupatta.
displayCategoryCount = duckdb.sql("SELECT category, COUNT(*) AS count_category FROM df GROUP BY category ORDER BY COUNT(*) DESC").df()
print(displayCategoryCount)

#Distribution of shipments based on state. Most common: maharashtra. Least common: NaN.
displayStateCount = duckdb.sql("SELECT ship_state, COUNT(*) AS count_state FROM df GROUP BY ship_state ORDER BY COUNT(*) DESC").df()
print(displayStateCount)

         date
0  2022-03-31
1  2022-04-01
2  2022-04-02
3  2022-04-03
4  2022-04-04
..        ...
86 2022-06-25
87 2022-06-26
88 2022-06-27
89 2022-06-28
90 2022-06-29

[91 rows x 1 columns]
                           status  count_status
0                         shipped         77804
1    shipped - delivered to buyer         28769
2                       cancelled         18332
3    shipped - returned to seller          1953
4             shipped - picked up           973
5                         pending           658
6   pending - waiting for pick up           281
7   shipped - returning to seller           145
8      shipped - out for delivery            35
9     shipped - rejected by buyer            11
10                       shipping             8
11      shipped - lost in transit             5
12              shipped - damaged             1
    size  count_size
0      M       22711
1      L       22132
2     XL       20876
3    XXL       18096
4      S       17090
5    3XL   

In [10]:
#Preliminary data assessment (continues):
#Distribution of shipments based on city. Most common: Bengaluru. Least common: lots of cities with one entry(?).
displayCityCount = duckdb.sql("SELECT ship_city, COUNT(*) AS count_city FROM df GROUP BY ship_city ORDER BY COUNT(*) DESC").df()
print(displayCityCount)

#One entry cities = 3327 ... too high!
displayCityNumbers = duckdb.sql("SELECT ship_city, COUNT(*) AS count_city FROM df GROUP BY ship_city HAVING count_city =1 ORDER BY ship_city ASC").df()
print(displayCityNumbers)


                ship_city  count_city
0               bengaluru       11900
1               hyderabad        9128
2                  mumbai        7124
3               new_delhi        6340
4                 chennai        6288
...                   ...         ...
7273            kalinagar           1
7274           vasai_(_w)           1
7275     kurseong_-734204           1
7276  nilje_dombivli_east           1
7277       pilkhuwa_hapur           1

[7278 rows x 2 columns]
                                           ship_city  count_city
0     (chikmagalur_disterict)._____(n.r_pur_thaluku)           1
1                     (via_cuncolim)quepem,south_goa           1
2                                         ,hyderabad           1
3                 ,raibarely_road_faizabad_(ayodhya)           1
4                                            ..katra           1
...                                              ...         ...
3322                                       zionpuram           1

In [9]:
#Preliminary data assessment questions:

#Which product styles and categories are the most popular in the city of Mumbai? western_dress, set and kurta in that order.
mumbaiPopularStyles = duckdb.sql("SELECT style, category, count(*) AS popularity FROM df WHERE ship_city = 'mumbai' GROUP BY style, category ORDER BY popularity DESC LIMIT 5").df()
print(mumbaiPopularStyles)

#What is the sales trend for the different months? Kurta and sets are always favored regardless of the month, not always in that order.
ordersByMonth = duckdb.sql("SELECT EXTRACT(MONTH FROM date) AS month, category, count(*) AS popularity FROM df GROUP BY month, category ORDER BY month, popularity DESC").df()
print(ordersByMonth)

#Orders by season(?) and popularity. In spring, sets and kurta are popular. In summer, kurta and sets.
seasonStyleCount = duckdb.sql("SELECT CASE WHEN MONTH(date) IN (3,4,5) THEN 'spring'"
              "WHEN MONTH(date) IN (6,7,8) THEN 'summer' END AS season, category, count(*) AS popularity "
              "FROM df GROUP BY season, category ORDER BY season, popularity DESC").df()
print(seasonStyleCount)

     style       category  popularity
0  JNE3797  western_dress         187
1   SET268            set         134
2    J0003            set         101
3    J0230            set          83
4  JNE3405          kurta          79
    month       category  popularity
0       3          kurta          76
1       3            set          75
2       3  western_dress           9
3       3            top           9
4       3         blouse           1
5       3   ethnic_dress           1
6       4            set       20204
7       4          kurta       19747
8       4  western_dress        4164
9       4            top        3928
10      4         blouse         418
11      4   ethnic_dress         351
12      4         bottom         180
13      4          saree          75
14      5            set       16016
15      5          kurta       14933
16      5  western_dress        6059
17      5            top        4055
18      5   ethnic_dress         451
19      5         blouse        

In [162]:
#Kimball start model implementation

#Dimension Tables hold context around the 'what'. They hold qualitative and descriptive variables => details.
#Ex. who? what? when? where? why? bought the coffee.

#Date Dimension Table
ducky.execute("""
    CREATE OR REPLACE TABLE dim_date (
        date_sk INT PRIMARY KEY,
        full_date TIMESTAMP,
        day_of_week INT,
        day_name VARCHAR,
        month_of_year INT,
        month_name VARCHAR,
        quarter INT,
        year INT,
        is_weekend BOOLEAN NOT NULL
    );
""")

#Generating dates
# ducky.execute("""
#     SELECT *
#     FROM generate_series(
#         DATE '2022-03-31',
#         DATE '2022-06-29',
#         INTERVAL 1 DAY
# ) AS dim_date(date_sk)
# """).fetchdf()

#Populate
ducky.execute("""
    INSERT INTO dim_date
    SELECT
        CAST(strftime('%Y%m%d', d.d) AS INT) AS date_sk,
        d AS full_date,
        dayofweek(d.d) AS day_of_week,
        dayname(d.d) AS day_name,
        month(d.d) AS month_of_year,
        monthname(d.d) AS month_name,
        quarter(d.d) AS quarter,
        year(d.d) AS year,
        CASE WHEN dayofweek(d.d) IN (0, 6) THEN true ELSE false END AS is_weekend
    FROM generate_series(
        DATE '2022-03-31',
        DATE '2022-06-29',
        INTERVAL 1 DAY
    ) AS d(d);
""")


# Make sure our dim_date table has the correct values
result_dim_date = ducky.execute("""
SELECT * FROM dim_date LIMIT 18
""").fetchdf()
print(result_dim_date)

     date_sk  full_date  day_of_week   day_name  month_of_year month_name  \
0   20220331 2022-03-31            4   Thursday              3      March   
1   20220401 2022-04-01            5     Friday              4      April   
2   20220402 2022-04-02            6   Saturday              4      April   
3   20220403 2022-04-03            0     Sunday              4      April   
4   20220404 2022-04-04            1     Monday              4      April   
5   20220405 2022-04-05            2    Tuesday              4      April   
6   20220406 2022-04-06            3  Wednesday              4      April   
7   20220407 2022-04-07            4   Thursday              4      April   
8   20220408 2022-04-08            5     Friday              4      April   
9   20220409 2022-04-09            6   Saturday              4      April   
10  20220410 2022-04-10            0     Sunday              4      April   
11  20220411 2022-04-11            1     Monday              4      April   

In [163]:
#Product Dimension Table
ducky.execute("CREATE OR REPLACE SEQUENCE seq_product_sk START 1")
ducky.execute("DROP TABLE IF EXISTS dim_product ")

#Create product dimension table
ducky.execute("""
    CREATE TABLE dim_product (
        product_sk INT PRIMARY KEY,
        asin VARCHAR,
        sku VARCHAR,
        style VARCHAR,
        category VARCHAR,
        size VARCHAR
    );
""")

#Work around as autoincrement does not work with duckDB(?)
ducky.execute("""
    ALTER TABLE dim_product
    ALTER COLUMN product_sk SET DEFAULT nextval('seq_product_sk');
""")

#Get values for city, state, zipcode and country from original df, clean them and then use them to populate similar named columns in dim_location
product_df = df[['asin', 'sku', 'style', 'category', 'size']].drop_duplicates()
products = [tuple(row) for row in product_df.itertuples(index=False, name=None)]
ducky.executemany(
    "INSERT INTO dim_product (asin, sku, style, category, size) VALUES (?, ?, ?, ?, ?)", products
)

#Make sure our dim_product table has the correct values
result_dim_product = ducky.execute("""
SELECT * FROM dim_product LIMIT 5
""").fetchdf()
print(result_dim_product)

   product_sk        asin              sku    style       category size
0           1  B09KXVBD7Z   SET389-KR-NP-S   SET389            Set    S
1           2  B09K3WFS32  JNE3781-KR-XXXL  JNE3781          kurta  3XL
2           3  B07WV4JV4D    JNE3371-KR-XL  JNE3371          kurta   XL
3           4  B099NRCT7B       J0341-DR-L    J0341  Western Dress    L
4           5  B098714BZP  JNE3671-TU-XXXL  JNE3671            Top  3XL


In [164]:
#Location Dimension Table
ducky.execute("CREATE OR REPLACE SEQUENCE seq_location_sk START 1")
ducky.execute("DROP TABLE IF EXISTS dim_location ")

#Create location dimension table
ducky.execute("""
    CREATE TABLE dim_location (
        location_sk INT PRIMARY KEY,
        city VARCHAR,
        state VARCHAR,
        zipcode VARCHAR,
        country VARCHAR
    );
""")

#Work around as autoincrement does not work with duckDB(?)
#Rewrite this later...
ducky.execute("""
    ALTER TABLE dim_location
    ALTER COLUMN location_sk SET DEFAULT nextval('seq_location_sk');
""")

#Get values for city, state, zipcode and country from original df, clean them and then use them to populate similar named columns in dim_location
location_df = df[['ship_city', 'ship_state', 'ship_postal_code', 'ship_country']].drop_duplicates()
locations = [tuple(row) for row in location_df.itertuples(index=False, name=None)]
ducky.executemany(
    "INSERT INTO dim_location (city, state, zipcode, country) VALUES (?, ?, ?, ?)", locations
)

#Make sure our dim_location table has the correct values
result_dim_location = ducky.execute("""
SELECT * FROM dim_location LIMIT 5
""").fetchdf()
print(result_dim_location)

   location_sk         city        state   zipcode country
0            1       MUMBAI  MAHARASHTRA  400081.0      IN
1            2    BENGALURU    KARNATAKA  560085.0      IN
2            3  NAVI MUMBAI  MAHARASHTRA  410210.0      IN
3            4   PUDUCHERRY   PUDUCHERRY  605008.0      IN
4            5      CHENNAI   TAMIL NADU  600073.0      IN


In [202]:
#Kimball start model implementation

#Fact table:
#"What is it being sold?" The table holds quantitative and measurable variables => numbers.
#Ex. buying a cup of coffee

#todo: change this to CREATE OR REPLACE?
#In case the tables already exist
ducky.execute("DROP TABLE IF EXISTS fct_orders")

#This is a fact table based on the orders placed at Fashionable e-commerce website
#Grain: One row per product line item in an order => one customer bought one quantity of one product

ducky.execute("CREATE OR REPLACE SEQUENCE seq_product START 1")
ducky.execute("CREATE OR REPLACE SEQUENCE seq_location START 1")


ducky.execute("""
    CREATE TABLE fct_orders (
        order_id VARCHAR,
        product_sk INT,
        location_sk INT,
        quantity BIGINT,
        amount DOUBLE,
        status VARCHAR
    );
""")

ducky.execute("""
    ALTER TABLE fct_orders
    ALTER COLUMN product_sk SET DEFAULT nextval('seq_product')
""")

ducky.execute("""
    ALTER TABLE fct_orders
    ALTER COLUMN location_sk SET DEFAULT nextval('seq_location')
""")

#Get values for city, state, zipcode and country from original df, clean them and then use them to populate similar named columns in dim_location
orders_df = df[['order_id', 'qty', 'amount', 'status']].drop_duplicates()
orders = [tuple(row) for row in orders_df.itertuples(index=False, name=None)]
ducky.executemany(
    "INSERT INTO fct_orders (order_id, quantity, amount, status) VALUES (?, ?, ?, ?)", orders
)
ducky.execute(
    "SELECT * FROM fct_orders JOIN dim_product USING(product_sk)"
)
ducky.execute(
    "SELECT * FROM fct_orders JOIN dim_location USING(location_sk)"
)

#Make sure our dim_location table has the correct values
result_orders = ducky.execute("""
SELECT * FROM fct_orders LIMIT 20
""").fetchdf()
print(result_orders)


               order_id  product_sk  location_sk  quantity  amount  \
0   405-8078784-5731545           1            1         0  647.62   
1   171-9198151-1101146           2            2         1  406.00   
2   404-0687676-7273146           3            3         1  329.00   
3   403-9615377-8133951           4            4         0  753.33   
4   407-1069790-7240320           5            5         1  574.00   
5   404-1490984-4578765           6            6         1  824.00   
6   408-5748499-6859555           7            7         1  653.00   
7   406-7807733-3785945           8            8         1  399.00   
8   407-5443024-5233168           9            9         0     NaN   
9   402-4393761-0311520          10           10         1  363.00   
10  407-5633625-6970741          11           11         1  685.00   
11  171-4638481-6326716          12           12         1  364.00   
12  405-5513694-8146768          13           13         1  399.00   
13  408-7955685-3083

In [ ]:

INTERMEDIATE LAYER IDEA?

SELECT index, order_id, style, sku, category, size, asin, qty AS quantity, amount,
FROM {{ ref('stg_orders')}}
